In [1]:
# -*- coding: utf-8 -*-
"""
eval_deepar_5min_local_model_on_cloud_mu_over_sigma.py

云端推理脚本：使用“本地 BigQuant e2e 5min 压缩 parquet 训练出的 DeepAR JSON 模型”，
在云端 DAI 5min 原始表上推理打分。

适配训练脚本：
    DeepAR 本地训练版：读取 BigQuant e2e 5min 压缩 parquet，
    训练 Student-t DeepAR，并保存真正 JSON 模型。

关键适配：
1. 本地训练：
   - 使用 instrument_id；
   - open/high/low/close/amount/bid_price1/ask_price1 从“分” /100 还原为“元”；
   - volume/bid_volume1/ask_volume1 不做 /100；
   - 特征：open/high/low/close/volume/amount/bid_volume1/ask_volume1/bid_price1/ask_price1；
   - DeepAR 额外输入 previous_available_trading_day_close_to_close_return；
   - target 为 next_trading_day_close_to_close_return；
   - checkpoint 为 JSON 文本。

2. 云端推理：
   - 使用云端 DAI 5min stock 表；
   - 使用 instrument 作为输出标识；
   - 云端原始表价格 / amount 通常已经是“元”，不要再 /100；
   - 对 volume/amount/bid_volume1/ask_volume1 做 log1p，与训练一致；
   - 使用 checkpoint 保存的 mean/std 标准化特征；
   - 使用 checkpoint 保存的 target_mean/target_std 标准化 prev_daily_return；
   - 模型输出 Student-t 的 loc、scale、df；
   - 将 loc、scale 反标准化到原始收益率尺度；
   - 将 Student-t scale 转换为预测标准差：
         pred_sigma = pred_scale * sqrt(df / (df - 2))
   - score 输出：
         score = pred_mu / pred_sigma

3. 输出：
      date, instrument, score

4. BigQuant DAI 分区表：
   - 查询 bigalpha_2026_stock_bar5m 等分区表时，必须显式传 filters 参数；
   - 本脚本所有 dai.query 均已使用 filters={"date": [...]}。
"""

from __future__ import annotations

import json
import logging
import os
import time
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 环境设置
# ============================================================

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"


# ============================================================
# 日志
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("infer_deepar_5m_local_on_cloud_mu_over_sigma")


def log_info(msg: str, **kwargs: Any) -> None:
    if kwargs:
        logger.info("%s | %s", msg, " | ".join(f"{k}={v}" for k, v in kwargs.items()))
    else:
        logger.info("%s", msg)


# ============================================================
# 与训练脚本保持一致的配置
# ============================================================

_HERE = os.getcwd()

MODEL_PATH = os.environ.get(
    "MODEL_PATH",
    os.path.join(
        _HERE,
        "stock_deepar_5min_to_daily_return_csi1000_local_e2e_v1.json",
    ),
)

HISTORY_BUFFER_DAYS = 10

TARGET_HORIZON_TRADING_DAYS = 1
TARGET_NAME = "next_trading_day_close_to_close_return"

PRICE_COLS = ["open", "high", "low", "close"]
VOL_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]
ORDERBOOK_PRICE_COLS = ["bid_price1", "ask_price1"]

FEATURE_COLS = PRICE_COLS + VOL_COLS + ORDERBOOK_PRICE_COLS
N_FEAT = len(FEATURE_COLS)

SEQ_LEN = 320

BATCH = 256

MODEL_CFG = {
    "n_feat": N_FEAT,
    "seq_len": SEQ_LEN,
    "encoder_hidden": 128,
    "encoder_layers": 2,
    "decoder_hidden": 128,
    "dropout": 0.25,
    "min_scale": 0.05,
    "min_df": 5.0,
}

DEEPAR_LAG_TARGET_NAME = "previous_available_trading_day_close_to_close_return"
DEEPAR_DISTRIBUTION = "StudentT"

LOCAL_ID_COL = "instrument_id"
CLOUD_ID_COL = "instrument"

LOCAL_COMPRESSED_PRICE_SCALE = 100.0
LOCAL_PRICE_SCALE_COLS = [
    "open",
    "high",
    "low",
    "close",
    "amount",
    "bid_price1",
    "ask_price1",
]

PRICE_NA_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]


# ============================================================
# mu / sigma 配置
# ============================================================

# 绝对下限，防止预测 sigma 过小导致 score 爆炸。
SIGMA_ABSOLUTE_FLOOR = float(os.environ.get("SIGMA_ABSOLUTE_FLOOR", "1e-8"))

# 相对下限：最终 sigma_floor 至少为中位数 sigma 的该比例。
SIGMA_MEDIAN_FLOOR_RATIO = float(
    os.environ.get("SIGMA_MEDIAN_FLOOR_RATIO", "1e-3")
)

if not np.isfinite(SIGMA_ABSOLUTE_FLOOR) or SIGMA_ABSOLUTE_FLOOR <= 0:
    raise ValueError("SIGMA_ABSOLUTE_FLOOR 必须为正的有限数。")

if (
    not np.isfinite(SIGMA_MEDIAN_FLOOR_RATIO)
    or SIGMA_MEDIAN_FLOOR_RATIO < 0
):
    raise ValueError("SIGMA_MEDIAN_FLOOR_RATIO 必须为非负有限数。")


# ============================================================
# 通用工具
# ============================================================

def _to_path_string(path_like) -> str:
    return os.fspath(path_like)


def _as_list(value) -> list:
    if value is None:
        return []
    return list(value)


def _unique_keep_order(items: Sequence[str]) -> List[str]:
    seen = set()
    out: List[str] = []

    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)

    return out


def read_end_timestamp(ed) -> pd.Timestamp:
    ts = pd.to_datetime(ed)

    if ts == ts.normalize():
        ts = ts + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)

    return ts


def daily_last_bar_positions(trade_days: np.ndarray) -> np.ndarray:
    if len(trade_days) == 0:
        return np.empty(0, dtype=np.int64)

    return np.flatnonzero(np.append(trade_days[1:] != trade_days[:-1], True))


def _sql_datetime_literal(ts: pd.Timestamp) -> str:
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%M:%S")


def _make_dai_date_filters(
    read_start_ts: pd.Timestamp,
    read_end_ts: pd.Timestamp,
) -> dict:
    """
    构造 BigQuant DAI 分区 filters。

    end 给到 read_end 所在日期的下一天，避免 filters 左闭右开时漏掉最后一天。
    SQL WHERE 仍会精确裁剪时间范围。
    """
    start_s = pd.to_datetime(read_start_ts).normalize().strftime("%Y-%m-%d")
    end_s = (
        pd.to_datetime(read_end_ts).normalize() + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")

    return {
        "date": [start_s, end_s],
    }


def _safe_table_name(table: str) -> str:
    table = str(table).strip()

    if not table:
        raise ValueError("云端行情表名为空。")

    if any(ch in table for ch in [";", "\n", "\r", "\t"]):
        raise ValueError(f"表名包含非法字符：{table!r}")

    return table


def _dai_query_df(sql: str, filters: Optional[dict] = None) -> pd.DataFrame:
    """
    兼容不同 BigQuant / DAI 环境的 query 返回对象。

    BigQuant 分区表即使 SQL 里有 WHERE date，也通常仍要求显式传：
        dai.query(sql, filters={"date": ["YYYY-MM-DD", "YYYY-MM-DD"]})
    """
    try:
        import dai
    except Exception as exc:
        raise ImportError(
            "当前环境无法 import dai。请确认脚本运行在 BigQuant 云端 DAI 环境中。"
        ) from exc

    if filters is None:
        result = dai.query(sql)
    else:
        result = dai.query(sql, filters=filters)

    if hasattr(result, "df"):
        df_attr = result.df
        return df_attr() if callable(df_attr) else df_attr

    if hasattr(result, "to_pandas"):
        return result.to_pandas()

    if isinstance(result, pd.DataFrame):
        return result

    raise TypeError(f"无法将 dai.query 返回对象转换为 DataFrame：{type(result)}")


# ============================================================
# 云端表字段解析
# ============================================================

def _cloud_feature_aliases() -> Dict[str, List[str]]:
    """
    目标列名 -> 云端表可能存在的候选列名。
    """
    aliases: Dict[str, List[str]] = {
        "date": ["date", "datetime", "trade_time", "trade_date"],
        "instrument": ["instrument", "instrument_id", "symbol", "code"],

        "open": ["open", "open_price"],
        "high": ["high", "high_price"],
        "low": ["low", "low_price"],
        "close": ["close", "close_price"],

        "volume": ["volume", "vol"],
        "amount": ["amount", "turnover", "turnover_value"],

        "bid_volume1": [
            "bid_volume1",
            "bid_volume_1",
            "bid_vol1",
            "bid_vol_1",
            "bid_size1",
            "bid_size_1",
            "bid1_volume",
        ],
        "ask_volume1": [
            "ask_volume1",
            "ask_volume_1",
            "ask_vol1",
            "ask_vol_1",
            "ask_size1",
            "ask_size_1",
            "ask1_volume",
        ],
        "bid_price1": [
            "bid_price1",
            "bid_price_1",
            "bid_px1",
            "bid_px_1",
            "bid1_price",
        ],
        "ask_price1": [
            "ask_price1",
            "ask_price_1",
            "ask_px1",
            "ask_px_1",
            "ask1_price",
        ],
    }

    return aliases


def inspect_cloud_columns(
    table: str,
    read_start=None,
    read_end=None,
) -> List[str]:
    """
    探测云端表字段。

    注意：
    BigQuant 分区表即使 SELECT * LIMIT 1，也要传 filters。
    """
    table = _safe_table_name(table)

    if read_start is None:
        read_start_ts = pd.Timestamp("2000-01-01")
    else:
        read_start_ts = pd.to_datetime(read_start)

    if read_end is None:
        read_end_ts = pd.Timestamp.today()
    else:
        read_end_ts = read_end_timestamp(read_end)

    filters = _make_dai_date_filters(read_start_ts, read_end_ts)

    sql = f"SELECT * FROM {table} LIMIT 1"

    df = _dai_query_df(sql, filters=filters)

    cols = [str(c) for c in df.columns]

    if not cols:
        raise RuntimeError(f"无法读取云端表字段：{table}")

    return cols


def resolve_cloud_column_map(
    table: str,
    read_start=None,
    read_end=None,
) -> Dict[str, str]:
    """
    返回：
        目标列名 -> 实际云端列名
    """
    cols = inspect_cloud_columns(
        table=table,
        read_start=read_start,
        read_end=read_end,
    )

    lower_to_real = {str(c).lower(): str(c) for c in cols}

    aliases = _cloud_feature_aliases()
    required_targets = _unique_keep_order(["date", "instrument", *FEATURE_COLS])

    out: Dict[str, str] = {}

    for target_col in required_targets:
        candidates = aliases.get(target_col, [target_col])
        found = None

        for cand in candidates:
            real = lower_to_real.get(str(cand).lower())

            if real is not None:
                found = real
                break

        if found is None:
            raise KeyError(
                "云端表缺少模型推理所需字段。\n"
                f"缺少目标字段：{target_col!r}\n"
                f"候选字段：{candidates}\n"
                f"云端表：{table}\n"
                f"当前表字段：{cols}\n\n"
                "请确认你传入的是 5min 表，并且包含："
                "open/high/low/close/volume/amount/"
                "bid_volume1/ask_volume1/bid_price1/ask_price1。"
            )

        out[target_col] = found

    return out


def build_select_exprs(column_map: Dict[str, str]) -> List[str]:
    exprs: List[str] = []
    required_targets = _unique_keep_order(["date", "instrument", *FEATURE_COLS])

    for target_col in required_targets:
        real_col = column_map[target_col]

        if real_col == target_col:
            exprs.append(real_col)
        else:
            exprs.append(f"{real_col} AS {target_col}")

    return exprs


def load_cloud_bars(
    table: str,
    read_start,
    read_end,
    instruments: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    """
    从云端 DAI 表读取 5min 行情与一档盘口数据。

    云端原始表通常已经是“元”，这里不做 /100。
    输出统一字段：
        date, instrument, FEATURE_COLS...
    """
    table = _safe_table_name(table)

    read_start_ts = pd.to_datetime(read_start)
    read_end_ts = read_end_timestamp(read_end)

    filters = _make_dai_date_filters(read_start_ts, read_end_ts)

    column_map = resolve_cloud_column_map(
        table=table,
        read_start=read_start_ts,
        read_end=read_end_ts,
    )

    date_col_real = column_map["date"]
    instrument_col_real = column_map["instrument"]
    select_exprs = build_select_exprs(column_map)

    where_parts = [
        f"{date_col_real} >= '{_sql_datetime_literal(read_start_ts)}'",
        f"{date_col_real} <= '{_sql_datetime_literal(read_end_ts)}'",
    ]

    if instruments is not None:
        inst_list = [
            str(x).strip()
            for x in instruments
            if str(x).strip()
        ]

        if not inst_list:
            raise RuntimeError("传入 instruments 为空，无法读取云端数据。")

        quoted = ",".join(
            "'" + x.replace("'", "''") + "'"
            for x in inst_list
        )
        where_parts.append(f"{instrument_col_real} IN ({quoted})")

    sql = f"""
        SELECT
            {", ".join(select_exprs)}
        FROM {table}
        WHERE {" AND ".join(where_parts)}
        ORDER BY {instrument_col_real}, {date_col_real}
    """

    log_info(
        "开始读取云端 5min DeepAR 推理数据",
        table=table,
        read_start=str(read_start_ts),
        read_end=str(read_end_ts),
        filters=filters,
        feature_count=len(FEATURE_COLS),
        instruments=("all" if instruments is None else len(instruments)),
    )

    df = _dai_query_df(sql, filters=filters)

    if df.empty:
        raise RuntimeError(
            "云端行情读取为空。\n"
            f"table={table}, read_start={read_start_ts}, "
            f"read_end={read_end_ts}, filters={filters}"
        )

    return df


def normalize_cloud_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    云端原始表通常已经是元，所以这里不做 /100。

    与训练保持一致：
    - open/high/low/close/bid_price1/ask_price1 中的 -1 视为缺失；
    - volume/amount/bid_volume1/ask_volume1 不设为缺失，
      后续 log1p(clip(lower=0))；
    - amount 在本地训练中 /100 后参与 log1p；
      云端通常已经是元，不再缩放。
    """
    df = df.copy()

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["instrument"] = df["instrument"].astype("string").str.strip()

    for col in FEATURE_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in PRICE_NA_COLS:
        if col in df.columns:
            df.loc[df[col] == -1, col] = np.nan

    return df


# ============================================================
# DeepAR 模型结构，与训练脚本一致
# ============================================================

class DeepARDailyReturn(nn.Module):
    def __init__(
        self,
        n_feat: int,
        seq_len: int = SEQ_LEN,
        encoder_hidden: int = 64,
        encoder_layers: int = 2,
        decoder_hidden: int = 64,
        dropout: float = 0.10,
        min_scale: float = 1e-4,
        min_df: float = 2.10,
    ) -> None:
        super().__init__()

        if n_feat <= 0 or seq_len <= 0:
            raise ValueError("n_feat 和 seq_len 必须大于 0。")

        if min_scale <= 0 or min_df <= 2:
            raise ValueError("min_scale 必须大于 0，min_df 必须大于 2。")

        self.n_feat = int(n_feat)
        self.seq_len = int(seq_len)
        self.encoder_hidden = int(encoder_hidden)
        self.decoder_hidden = int(decoder_hidden)
        self.min_scale = float(min_scale)
        self.min_df = float(min_df)

        self.encoder = nn.LSTM(
            input_size=self.n_feat,
            hidden_size=self.encoder_hidden,
            num_layers=int(encoder_layers),
            batch_first=True,
            dropout=float(dropout) if int(encoder_layers) > 1 else 0.0,
        )

        self.context_projection = nn.Sequential(
            nn.Linear(self.encoder_hidden, self.decoder_hidden),
            nn.Tanh(),
        )

        self.decoder_cell = nn.GRUCell(
            input_size=self.decoder_hidden + 1,
            hidden_size=self.decoder_hidden,
        )

        self.dropout = nn.Dropout(float(dropout))

        self.distribution_head = nn.Sequential(
            nn.LayerNorm(self.decoder_hidden),
            nn.Linear(self.decoder_hidden, self.decoder_hidden),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(self.decoder_hidden, 3),
        )

    def _validate_inputs(
        self,
        x: torch.Tensor,
        prev_daily_return: torch.Tensor,
    ) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"x 必须为三维，实际形状={tuple(x.shape)}")

        b, l, f = x.shape

        if l != self.seq_len:
            raise ValueError(
                f"输入序列长度必须为 {self.seq_len}，实际为 {l}"
            )

        if f != self.n_feat:
            raise ValueError(
                f"输入特征维度错误：模型要求 {self.n_feat}，实际收到 {f}"
            )

        prev = prev_daily_return.reshape(-1)

        if len(prev) != b:
            raise ValueError(
                "prev_daily_return batch 维度与 x 不一致："
                f"{len(prev)} vs {b}"
            )

        return prev

    def forward(
        self,
        x: torch.Tensor,
        prev_daily_return: torch.Tensor,
    ):
        prev = self._validate_inputs(x, prev_daily_return)

        encoded, _ = self.encoder(x)
        context = self.dropout(
            self.context_projection(encoded[:, -1, :])
        )

        decoder_input = torch.cat(
            [context, prev.unsqueeze(-1)],
            dim=-1,
        )

        decoder_state = self.decoder_cell(
            decoder_input,
            context,
        )

        raw = self.distribution_head(
            self.dropout(decoder_state)
        )

        loc = raw[:, 0]
        scale = F.softplus(raw[:, 1]) + self.min_scale
        df = F.softplus(raw[:, 2]) + self.min_df

        return loc, scale, df

    def distribution(
        self,
        x,
        prev_daily_return,
    ):
        loc, scale, df = self.forward(x, prev_daily_return)

        return torch.distributions.StudentT(
            df=df,
            loc=loc,
            scale=scale,
        )

    def negative_log_likelihood(
        self,
        x,
        prev_daily_return,
        target_return,
    ):
        target = target_return.reshape(-1)
        nll = -self.distribution(
            x,
            prev_daily_return,
        ).log_prob(target)

        if not torch.isfinite(nll).all():
            raise FloatingPointError(
                "DeepAR Student-t 负对数似然出现 NaN 或无穷值。"
            )

        return nll.mean()


# ============================================================
# JSON checkpoint 加载
# ============================================================

def _json_item_to_tensor(item: dict) -> torch.Tensor:
    dtype_map = {
        "float16": torch.float16,
        "float32": torch.float32,
        "float64": torch.float64,
        "int8": torch.int8,
        "int16": torch.int16,
        "int32": torch.int32,
        "int64": torch.int64,
        "uint8": torch.uint8,
        "bool": torch.bool,
    }

    dtype = item["dtype"]

    if dtype not in dtype_map:
        raise ValueError(f"不支持的 tensor dtype: {dtype}")

    return torch.tensor(
        item["data"],
        dtype=dtype_map[dtype],
    ).reshape(tuple(item["shape"]))


def _state_dict_from_jsonable(state_dict_json: dict) -> dict:
    return {
        k: _json_item_to_tensor(v)
        for k, v in state_dict_json.items()
    }


def load_json_checkpoint(model_path) -> dict:
    model_path = os.fspath(model_path)

    with open(model_path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    obj = dict(obj)
    obj["state_dict"] = _state_dict_from_jsonable(
        obj["state_dict"]
    )

    return obj


# ============================================================
# checkpoint 兼容性检查
# ============================================================

def validate_checkpoint_compatibility(ckpt: dict) -> None:
    required = {
        "state_dict",
        "model_cfg",
        "model_type",
        "distribution",
        "lag_target_name",
        "feature_cols",
        "seq_len",
        "mean",
        "std",
        "target_mean",
        "target_std",
        "target_clip_lower",
        "target_clip_upper",
        "target_name",
        "target_horizon_trading_days",
    }

    missing = required - set(ckpt.keys())

    if missing:
        raise KeyError(
            f"DeepAR 模型文件缺少必要内容：{sorted(missing)}"
        )

    if ckpt.get("model_type") != "DeepARDailyReturn":
        raise ValueError(
            "checkpoint 不是 DeepARDailyReturn 模型。\n"
            f"模型保存值={ckpt.get('model_type')!r}"
        )

    if ckpt.get("distribution") != DEEPAR_DISTRIBUTION:
        raise ValueError(
            "checkpoint 的预测分布不是预期的 Student-t。\n"
            f"当前={DEEPAR_DISTRIBUTION!r}, "
            f"模型={ckpt.get('distribution')!r}"
        )

    if ckpt.get("lag_target_name") != DEEPAR_LAG_TARGET_NAME:
        raise ValueError(
            "checkpoint 的 lag target 与当前脚本不一致。\n"
            f"当前={DEEPAR_LAG_TARGET_NAME!r}, "
            f"模型={ckpt.get('lag_target_name')!r}"
        )

    if _as_list(ckpt.get("feature_cols")) != list(FEATURE_COLS):
        raise ValueError(
            "FEATURE_COLS 不一致。\n"
            f"当前={FEATURE_COLS}\n"
            f"模型={ckpt.get('feature_cols')}"
        )

    if int(ckpt.get("seq_len")) != SEQ_LEN:
        raise ValueError(
            "SEQ_LEN 不一致。\n"
            f"当前={SEQ_LEN}\n"
            f"模型={ckpt.get('seq_len')}"
        )

    if ckpt.get("target_name") != TARGET_NAME:
        raise ValueError(
            "target_name 不一致。\n"
            f"当前={TARGET_NAME!r}\n"
            f"模型={ckpt.get('target_name')!r}"
        )

    if (
        int(ckpt.get("target_horizon_trading_days"))
        != TARGET_HORIZON_TRADING_DAYS
    ):
        raise ValueError(
            "target_horizon_trading_days 不一致。\n"
            f"当前={TARGET_HORIZON_TRADING_DAYS}\n"
            f"模型={ckpt.get('target_horizon_trading_days')!r}"
        )

    saved_id_col = ckpt.get("id_col")

    if saved_id_col not in {LOCAL_ID_COL, CLOUD_ID_COL, None}:
        raise ValueError(
            "模型 id_col 不在兼容范围内。\n"
            "允许 instrument_id / instrument / None，"
            f"模型 id_col={saved_id_col!r}"
        )

    if saved_id_col == LOCAL_ID_COL:
        log_info(
            "检测到本地训练模型",
            id_col=saved_id_col,
            note=(
                "云端推理会使用 instrument 作为输出标识；"
                "id_col 不参与模型输入。"
            ),
        )
    elif saved_id_col == CLOUD_ID_COL:
        log_info(
            "检测到云端训练模型",
            id_col=saved_id_col,
        )

    saved_model_cfg = ckpt.get("model_cfg", {})

    expected_cfg_items = {
        "n_feat": N_FEAT,
        "seq_len": SEQ_LEN,
    }

    for key, expected_value in expected_cfg_items.items():
        saved_value = saved_model_cfg.get(key)

        if saved_value != expected_value:
            raise ValueError(
                f"模型配置 {key} 不匹配。\n"
                f"当前={expected_value}\n"
                f"模型={saved_value}"
            )

    mean = np.asarray(
        ckpt["mean"],
        dtype=np.float32,
    )
    std = np.asarray(
        ckpt["std"],
        dtype=np.float32,
    )

    if mean.shape != (N_FEAT,):
        raise ValueError(
            "mean 维度错误。\n"
            f"期望={(N_FEAT,)}\n"
            f"实际={mean.shape}"
        )

    if std.shape != (N_FEAT,):
        raise ValueError(
            "std 维度错误。\n"
            f"期望={(N_FEAT,)}\n"
            f"实际={std.shape}"
        )

    if (
        not np.isfinite(mean).all()
        or not np.isfinite(std).all()
    ):
        raise ValueError(
            "模型保存的 mean/std 包含 NaN 或无穷值。"
        )

    if np.any(std <= 0):
        raise ValueError(
            "模型保存的 std 必须全部大于 0。"
        )

    for key in [
        "target_clip_lower",
        "target_clip_upper",
        "target_mean",
        "target_std",
    ]:
        value = float(ckpt[key])

        if not np.isfinite(value):
            raise ValueError(
                f"{key} 非有限值：{value}"
            )

    if float(ckpt["target_std"]) <= 0:
        raise ValueError(
            "target_std 必须大于 0。"
        )

    bar_interval = ckpt.get("bar_interval")

    if bar_interval not in {"5min", "5m", None}:
        raise ValueError(
            "模型 bar_interval 不是 5min，不能用于当前 5min 云端推理脚本。\n"
            f"模型 bar_interval={bar_interval!r}"
        )

    saved_scale = ckpt.get(
        "local_compressed_price_scale"
    )

    if saved_scale is not None:
        log_info(
            "模型来自本地压缩数据训练",
            local_compressed_price_scale=saved_scale,
            note=(
                "训练脚本已将本地 "
                "open/high/low/close/amount/bid_price1/ask_price1 "
                "/100 还原为元；"
                "云端 stock 表通常已经是元，因此推理时不再 /100。"
            ),
        )


def load_model(
    model_path=MODEL_PATH,
    device: Optional[torch.device] = None,
):
    model_path = os.fspath(model_path)

    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"模型文件不存在：{model_path}\n"
            "请把本地训练脚本保存出的 JSON 模型文件"
            "和本推理脚本一起上传到云端。"
        )

    if device is None:
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

    ckpt = load_json_checkpoint(model_path)
    validate_checkpoint_compatibility(ckpt)

    model = DeepARDailyReturn(
        **ckpt["model_cfg"]
    ).to(device)

    model.load_state_dict(
        ckpt["state_dict"]
    )
    model.eval()

    return model, ckpt


# ============================================================
# 标准化 / 反标准化
# ============================================================

def apply_feature_stats(
    X: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    mean = np.asarray(
        mean,
        dtype=np.float32,
    )
    std = np.asarray(
        std,
        dtype=np.float32,
    )

    if (
        mean.shape != (N_FEAT,)
        or std.shape != (N_FEAT,)
    ):
        raise ValueError(
            "mean/std 维度错误："
            f"mean={mean.shape}, std={std.shape}, N_FEAT={N_FEAT}"
        )

    if (
        not np.isfinite(mean).all()
        or not np.isfinite(std).all()
        or np.any(std <= 0)
    ):
        raise ValueError(
            "特征标准化统计量非法。"
        )

    X = X.astype(
        np.float32,
        copy=False,
    )

    X -= mean.reshape(1, 1, -1)
    X /= std.reshape(1, 1, -1)

    return X.astype(
        np.float32,
        copy=False,
    )


def normalize_target_values(
    values,
    target_mean: float,
    target_std: float,
    lower_bound: float,
    upper_bound: float,
) -> np.ndarray:
    arr = np.asarray(
        values,
        dtype=np.float32,
    )

    if not np.isfinite(arr).all():
        raise ValueError(
            "待标准化目标包含 NaN 或无穷值。"
        )

    if (
        not np.isfinite(target_std)
        or target_std <= 0
    ):
        raise ValueError(
            "target_std 必须为正的有限数。"
        )

    arr = np.clip(
        arr,
        lower_bound,
        upper_bound,
    )

    return (
        (arr - np.float32(target_mean))
        / np.float32(target_std)
    ).astype(np.float32)


def denormalize_loc_scale(
    loc: np.ndarray,
    scale: np.ndarray,
    target_mean: float,
    target_std: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    将标准化目标空间中的 Student-t loc / scale
    映射回原始收益率空间。

    对 y_raw = y_norm * target_std + target_mean：
        loc_raw   = loc_norm * target_std + target_mean
        scale_raw = scale_norm * target_std
    """
    loc = np.asarray(
        loc,
        dtype=np.float64,
    )
    scale = np.asarray(
        scale,
        dtype=np.float64,
    )

    if (
        not np.isfinite(target_mean)
        or not np.isfinite(target_std)
        or target_std <= 0
    ):
        raise ValueError(
            "target_mean/target_std 非法。"
        )

    pred_mu = (
        loc * float(target_std)
        + float(target_mean)
    )

    pred_scale = (
        scale * float(target_std)
    )

    return pred_mu, pred_scale


def student_t_scale_to_sigma(
    scale: np.ndarray,
    df: np.ndarray,
) -> np.ndarray:
    """
    将 Student-t 分布的 scale 参数转换为标准差。

    当 df > 2 时：
        sigma = scale * sqrt(df / (df - 2))

    注意：
    Student-t 的 scale 不是标准差。
    """
    scale = np.asarray(
        scale,
        dtype=np.float64,
    )
    df = np.asarray(
        df,
        dtype=np.float64,
    )

    if scale.shape != df.shape:
        raise ValueError(
            "scale 与 df 形状不一致："
            f"scale={scale.shape}, df={df.shape}"
        )

    if not np.isfinite(scale).all():
        raise FloatingPointError(
            "预测 scale 包含 NaN 或无穷值。"
        )

    if not np.isfinite(df).all():
        raise FloatingPointError(
            "预测 df 包含 NaN 或无穷值。"
        )

    if np.any(scale <= 0):
        raise FloatingPointError(
            "预测 scale 存在非正值。"
        )

    if np.any(df <= 2.0):
        raise FloatingPointError(
            "预测 df 存在 <= 2 的值，Student-t 方差不存在。"
            f"最小 df={float(np.min(df))}"
        )

    sigma = scale * np.sqrt(
        df / (df - 2.0)
    )

    if not np.isfinite(sigma).all():
        raise FloatingPointError(
            "预测 sigma 包含 NaN 或无穷值。"
        )

    if np.any(sigma <= 0):
        raise FloatingPointError(
            "预测 sigma 存在非正值。"
        )

    return sigma


def build_mu_over_sigma_score(
    pred_mu: np.ndarray,
    pred_sigma: np.ndarray,
) -> Tuple[np.ndarray, float]:
    """
    构造：
        score = pred_mu / pred_sigma

    为防止极小 sigma 导致 score 爆炸，使用：
        sigma_floor = max(
            SIGMA_ABSOLUTE_FLOOR,
            median(pred_sigma) * SIGMA_MEDIAN_FLOOR_RATIO
        )
    """
    pred_mu = np.asarray(
        pred_mu,
        dtype=np.float64,
    )
    pred_sigma = np.asarray(
        pred_sigma,
        dtype=np.float64,
    )

    if pred_mu.shape != pred_sigma.shape:
        raise ValueError(
            "pred_mu 与 pred_sigma 形状不一致："
            f"pred_mu={pred_mu.shape}, pred_sigma={pred_sigma.shape}"
        )

    if not np.isfinite(pred_mu).all():
        raise FloatingPointError(
            "pred_mu 包含 NaN 或无穷值。"
        )

    if not np.isfinite(pred_sigma).all():
        raise FloatingPointError(
            "pred_sigma 包含 NaN 或无穷值。"
        )

    if np.any(pred_sigma <= 0):
        raise FloatingPointError(
            "pred_sigma 存在非正值。"
        )

    sigma_median = float(
        np.median(pred_sigma)
    )

    if (
        not np.isfinite(sigma_median)
        or sigma_median <= 0
    ):
        raise FloatingPointError(
            f"pred_sigma 中位数非法：{sigma_median}"
        )

    sigma_floor = max(
        SIGMA_ABSOLUTE_FLOOR,
        sigma_median * SIGMA_MEDIAN_FLOOR_RATIO,
    )

    safe_sigma = np.maximum(
        pred_sigma,
        sigma_floor,
    )

    score = pred_mu / safe_sigma

    if not np.isfinite(score).all():
        raise FloatingPointError(
            "mu_over_sigma score 包含 NaN 或无穷值。"
        )

    return score.astype(np.float64), float(sigma_floor)


# ============================================================
# 云端 DeepAR 推理数据集构建
# ============================================================

def build_deepar_dataset_cloud(
    table: str,
    sd,
    ed,
    mode: str = "infer",
    instruments: Optional[Sequence[str]] = None,
    stats: Optional[Tuple[np.ndarray, np.ndarray]] = None,
):
    """
    构建云端 DeepAR infer 数据集。

    与训练脚本 build_deepar_dataset 的 infer 模式保持一致：
    - 向前读取 HISTORY_BUFFER_DAYS；
    - 每个 instrument 每个交易日取最后一根 5min bar 为样本截点；
    - 输入窗口为过去 SEQ_LEN=240 根 5min bars；
    - prev_daily_return = close[t] / close[t-1] - 1；
    - X 用 checkpoint mean/std 标准化；
    - prev_daily_return 在 main 中用 target_mean/target_std 标准化。
    """
    if mode != "infer":
        raise ValueError(
            "云端推理脚本只支持 mode='infer'。"
        )

    if stats is None:
        raise ValueError(
            "infer 模式必须传入训练阶段保存的 stats=(mean,std)。"
        )

    if TARGET_HORIZON_TRADING_DAYS != 1:
        raise ValueError(
            "当前脚本仅支持下一交易日日收益率目标。"
        )

    t0 = time.time()

    sd_ts = pd.to_datetime(
        sd
    ).normalize()

    ed_ts = pd.to_datetime(
        ed
    ).normalize()

    if ed_ts < sd_ts:
        raise ValueError(
            "ed 必须不早于 sd。"
        )

    buffer_start = (
        sd_ts - pd.Timedelta(days=HISTORY_BUFFER_DAYS)
    ).strftime("%Y-%m-%d")

    raw_df = load_cloud_bars(
        table=table,
        read_start=buffer_start,
        read_end=ed,
        instruments=instruments,
    )

    df = normalize_cloud_features(
        raw_df
    )

    df = (
        df.replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "date",
                "instrument",
                *FEATURE_COLS,
            ]
        )
        .loc[
            lambda frame:
            frame["instrument"].ne("")
        ]
        .sort_values(
            ["instrument", "date"]
        )
        .drop_duplicates(
            ["instrument", "date"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise RuntimeError(
            "云端行情清洗后为空。"
        )

    # 与训练一致：对 VOL_COLS 做 log1p(clip(lower=0))
    for col in VOL_COLS:
        df[col] = np.log1p(
            df[col].clip(lower=0)
        )

    windows: List[np.ndarray] = []
    prev_returns: List[np.float32] = []
    keys: List[Tuple[pd.Timestamp, str]] = []

    instrument_count = 0
    insufficient_history = 0
    insufficient_lag_history = 0
    invalid_return_samples = 0
    invalid_window = 0

    for instrument, sub in df.groupby(
        "instrument",
        sort=False,
    ):
        instrument_count += 1

        sub = (
            sub.sort_values("date")
            .reset_index(drop=True)
        )

        if len(sub) < SEQ_LEN:
            insufficient_history += 1
            continue

        feats = sub[
            FEATURE_COLS
        ].to_numpy(dtype=np.float32)

        trade_days = (
            pd.to_datetime(
                sub["date"],
                errors="coerce",
            )
            .dt.normalize()
            .to_numpy(dtype="datetime64[ns]")
        )

        close_positions = daily_last_bar_positions(
            trade_days
        )

        if len(close_positions) < 2:
            insufficient_lag_history += 1
            continue

        daily_dates = trade_days[
            close_positions
        ]

        daily_close = sub[
            "close"
        ].to_numpy(dtype=np.float64)[
            close_positions
        ]

        for day_index, position in enumerate(
            close_positions
        ):
            sample_date = pd.Timestamp(
                daily_dates[day_index]
            ).normalize()

            if (
                sample_date < sd_ts
                or sample_date > ed_ts
            ):
                continue

            if position + 1 < SEQ_LEN:
                insufficient_history += 1
                continue

            if day_index < 1:
                insufficient_lag_history += 1
                continue

            previous_close = daily_close[
                day_index - 1
            ]
            current_close = daily_close[
                day_index
            ]

            if (
                previous_close <= 0
                or current_close <= 0
            ):
                invalid_return_samples += 1
                continue

            prev_daily_return = (
                current_close / previous_close
                - 1.0
            )

            if not np.isfinite(
                prev_daily_return
            ):
                invalid_return_samples += 1
                continue

            window = feats[
                position - SEQ_LEN + 1:
                position + 1
            ]

            if window.shape != (
                SEQ_LEN,
                N_FEAT,
            ):
                invalid_window += 1
                continue

            if not np.isfinite(
                window
            ).all():
                invalid_window += 1
                continue

            windows.append(window)
            prev_returns.append(
                np.float32(prev_daily_return)
            )
            keys.append(
                (sample_date, str(instrument))
            )

    if not windows:
        raise RuntimeError(
            "build_deepar_dataset_cloud 无推理样本。\n"
            f"table={table}, sd={sd}, ed={ed}, "
            f"股票数={instrument_count}, "
            f"分钟历史不足={insufficient_history}, "
            f"滞后收益历史不足={insufficient_lag_history}, "
            f"无效收益样本={invalid_return_samples}, "
            f"无效窗口={invalid_window}"
        )

    X_raw = np.stack(
        windows,
        axis=0,
    ).astype(np.float32)

    windows.clear()

    mean, std = stats

    X = apply_feature_stats(
        X_raw,
        mean=mean,
        std=std,
    )

    prev = np.asarray(
        prev_returns,
        dtype=np.float32,
    )

    idx_df = pd.DataFrame(
        keys,
        columns=[
            "date",
            "instrument",
        ],
    )

    idx_df["date"] = pd.to_datetime(
        idx_df["date"],
        errors="coerce",
    ).dt.normalize()

    idx_df["instrument"] = (
        idx_df["instrument"]
        .astype("string")
        .str.strip()
    )

    log_info(
        "DeepAR 云端 infer 集构建完成",
        samples=len(idx_df),
        days=idx_df["date"].nunique(),
        instruments=idx_df["instrument"].nunique(),
        x_shape=tuple(X.shape),
        prev_daily_return_mean=float(prev.mean()),
        prev_daily_return_std=float(prev.std()),
        elapsed=round(time.time() - t0, 2),
    )

    return X, prev, None, idx_df, (mean, std)


# ============================================================
# 数据源解析
# ============================================================

def resolve_bar5m_table(
    datasources: dict,
) -> str:
    """
    优先级：
    1. 环境变量 BAR5M_TABLE；
    2. datasources["bar5m"]；
    3. datasources["bar5min"]；
    4. datasources["stock_bar5m"]；
    5. datasources["bar"]。

    如果只传 bar30m，则报错，因为当前模型是 5min 模型。
    """
    env_table = os.environ.get(
        "BAR5M_TABLE"
    )

    if env_table:
        return _safe_table_name(
            env_table
        )

    for key in [
        "bar5m",
        "bar5min",
        "stock_bar5m",
        "bar",
    ]:
        if (
            key in datasources
            and datasources[key]
        ):
            return _safe_table_name(
                datasources[key]
            )

    if "bar30m" in datasources:
        raise ValueError(
            "当前 DeepAR 模型使用 5min 数据训练，"
            "不能直接使用 bar30m 表推理。\n"
            "请在 datasources 中传入 "
            "{'bar5m': '你的云端5min表名'}。"
        )

    raise KeyError(
        "datasources 中没有找到 5min 行情表。\n"
        "请使用例如："
        "datasources = {'bar5m': 'bigalpha_2026_stock_bar5m'}"
    )


# ============================================================
# 平台调用入口
# ============================================================

def main(
    datasources,
    start_date,
    end_date,
):
    """
    加载本地训练出的 DeepAR JSON 模型，
    在云端 DAI 5min 测试区间上推理打分。

    参数：
        datasources:
            例如：
                {"bar5m": "bigalpha_2026_stock_bar5m"}

        start_date:
            平台注入的测试开始时间

        end_date:
            平台注入的测试结束时间

    返回：
        DataFrame，字段必须为：
            date, instrument, score

        score 定义为：
            pred_mu / pred_sigma

        其中：
            pred_mu 为原始收益率尺度上的 Student-t loc；
            pred_sigma 为原始收益率尺度上的 Student-t 标准差：
                pred_scale * sqrt(df / (df - 2))
    """
    table = resolve_bar5m_table(
        datasources
    )

    official_start = pd.to_datetime(
        start_date,
        errors="coerce",
    ).normalize()

    official_end = pd.to_datetime(
        end_date,
        errors="coerce",
    ).normalize()

    if (
        pd.isna(official_start)
        or pd.isna(official_end)
    ):
        raise ValueError(
            "start_date 或 end_date 无法解析："
            f"start_date={start_date!r}, "
            f"end_date={end_date!r}"
        )

    if official_end < official_start:
        raise ValueError(
            "end_date 不能早于 start_date。"
        )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model_path_string = _to_path_string(
        MODEL_PATH
    )

    model, checkpoint = load_model(
        model_path=model_path_string,
        device=device,
    )

    mean = np.asarray(
        checkpoint["mean"],
        dtype=np.float32,
    )

    std = np.asarray(
        checkpoint["std"],
        dtype=np.float32,
    )

    target_mean = float(
        checkpoint["target_mean"]
    )
    target_std = float(
        checkpoint["target_std"]
    )
    target_clip_lower = float(
        checkpoint["target_clip_lower"]
    )
    target_clip_upper = float(
        checkpoint["target_clip_upper"]
    )

    log_info(
        "已加载本地训练 DeepAR JSON 模型，准备在云端 5min 原始表推理",
        path=model_path_string,
        device=str(device),
        table=table,
        start=str(start_date),
        end=str(end_date),
        feature_cols=FEATURE_COLS,
        seq_len=SEQ_LEN,
        distribution=DEEPAR_DISTRIBUTION,
        score_definition="pred_mu / pred_sigma",
        sigma_absolute_floor=SIGMA_ABSOLUTE_FLOOR,
        sigma_median_floor_ratio=SIGMA_MEDIAN_FLOOR_RATIO,
    )

    Xte, prev_raw, _, idx_df, _ = build_deepar_dataset_cloud(
        table=table,
        sd=start_date,
        ed=end_date,
        mode="infer",
        instruments=None,
        stats=(mean, std),
    )

    if (
        idx_df is None
        or idx_df.empty
    ):
        raise RuntimeError(
            "推理集为空，无法生成分数。"
        )

    if (
        len(Xte) != len(idx_df)
        or len(prev_raw) != len(idx_df)
    ):
        raise RuntimeError(
            "Xte / prev_raw / idx_df 长度不一致："
            f"len(Xte)={len(Xte)}, "
            f"len(prev_raw)={len(prev_raw)}, "
            f"len(idx_df)={len(idx_df)}"
        )

    prev_norm = normalize_target_values(
        values=prev_raw,
        target_mean=target_mean,
        target_std=target_std,
        lower_bound=target_clip_lower,
        upper_bound=target_clip_upper,
    )

    model.eval()

    Xte_t = torch.from_numpy(
        Xte.astype(
            np.float32,
            copy=False,
        )
    )

    prev_t = torch.from_numpy(
        prev_norm.astype(
            np.float32,
            copy=False,
        )
    )

    loc_parts: List[np.ndarray] = []
    scale_parts: List[np.ndarray] = []
    df_parts: List[np.ndarray] = []

    with torch.no_grad():
        for i in range(
            0,
            len(idx_df),
            BATCH,
        ):
            xb = Xte_t[
                i:i + BATCH
            ].to(
                device,
                non_blocking=(
                    device.type == "cuda"
                ),
            )

            pb = prev_t[
                i:i + BATCH
            ].to(
                device,
                non_blocking=(
                    device.type == "cuda"
                ),
            )

            loc, scale, df_pred_batch = model(
                xb,
                pb,
            )

            loc_parts.append(
                loc.detach().cpu().numpy()
            )
            scale_parts.append(
                scale.detach().cpu().numpy()
            )
            df_parts.append(
                df_pred_batch.detach().cpu().numpy()
            )

    loc_norm = np.concatenate(
        loc_parts,
        axis=0,
    ).astype(np.float64)

    scale_norm = np.concatenate(
        scale_parts,
        axis=0,
    ).astype(np.float64)

    df_pred = np.concatenate(
        df_parts,
        axis=0,
    ).astype(np.float64)

    if not (
        len(loc_norm)
        == len(scale_norm)
        == len(df_pred)
        == len(idx_df)
    ):
        raise RuntimeError(
            "模型输出长度不一致："
            f"loc={len(loc_norm)}, "
            f"scale={len(scale_norm)}, "
            f"df={len(df_pred)}, "
            f"idx={len(idx_df)}"
        )

    # 反标准化到原始下一交易日日收益率尺度。
    pred_mu, pred_scale = denormalize_loc_scale(
        loc=loc_norm,
        scale=scale_norm,
        target_mean=target_mean,
        target_std=target_std,
    )

    # Student-t 的 scale 参数不等于标准差。
    # 当 df > 2 时：
    #     pred_sigma = pred_scale * sqrt(df / (df - 2))
    pred_sigma = student_t_scale_to_sigma(
        scale=pred_scale,
        df=df_pred,
    )

    # 最终 score：
    #     score = pred_mu / pred_sigma
    score, sigma_floor = build_mu_over_sigma_score(
        pred_mu=pred_mu,
        pred_sigma=pred_sigma,
    )

    result = idx_df.copy()

    if "date" not in result.columns:
        raise KeyError(
            "idx_df 缺少 date 字段。"
        )

    if "instrument" not in result.columns:
        raise KeyError(
            "idx_df 缺少 instrument 字段。"
        )

    result["date"] = pd.to_datetime(
        result["date"],
        errors="coerce",
    ).dt.normalize()

    result["instrument"] = (
        result["instrument"]
        .astype("string")
        .str.strip()
    )

    result["score"] = score.astype(
        np.float64
    )

    result = (
        result.replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=[
                "date",
                "instrument",
                "score",
            ]
        )
        .loc[
            lambda frame:
            frame["instrument"].ne("")
        ]
        .drop_duplicates(
            ["date", "instrument"],
            keep="last",
        )
        [
            ["date", "instrument", "score"]
        ]
        .sort_values(
            ["date", "instrument"]
        )
        .reset_index(drop=True)
    )

    # 再保险：只返回官方评估窗口内的数据。
    result = result.loc[
        (result["date"] >= official_start)
        & (result["date"] <= official_end)
    ].copy()

    result = (
        result[
            ["date", "instrument", "score"]
        ]
        .sort_values(
            ["date", "instrument"]
        )
        .reset_index(drop=True)
    )

    if result.empty:
        raise RuntimeError(
            "裁剪到官方评估窗口后 result 为空。\n"
            f"start_date={official_start.date()}, "
            f"end_date={official_end.date()}"
        )

    daily_count = (
        result.groupby("date")["instrument"]
        .nunique()
        .sort_index()
    )

    log_info(
        "mu_over_sigma 分数构建完成",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
        first_date=str(
            result["date"].min().date()
        ),
        last_date=str(
            result["date"].max().date()
        ),
        min_daily_count=int(
            daily_count.min()
        ),
        median_daily_count=float(
            daily_count.median()
        ),
        max_daily_count=int(
            daily_count.max()
        ),
        score_mean=float(
            result["score"].mean()
        ),
        score_std=float(
            result["score"].std(ddof=0)
        ),
        score_min=float(
            result["score"].min()
        ),
        score_max=float(
            result["score"].max()
        ),
        loc_norm_mean=float(
            np.mean(loc_norm)
        ),
        loc_norm_std=float(
            np.std(loc_norm)
        ),
        scale_norm_mean=float(
            np.mean(scale_norm)
        ),
        pred_mu_mean=float(
            np.mean(pred_mu)
        ),
        pred_mu_std=float(
            np.std(pred_mu)
        ),
        pred_scale_mean=float(
            np.mean(pred_scale)
        ),
        pred_scale_median=float(
            np.median(pred_scale)
        ),
        pred_sigma_mean=float(
            np.mean(pred_sigma)
        ),
        pred_sigma_median=float(
            np.median(pred_sigma)
        ),
        pred_sigma_min=float(
            np.min(pred_sigma)
        ),
        pred_sigma_max=float(
            np.max(pred_sigma)
        ),
        pred_df_mean=float(
            np.mean(df_pred)
        ),
        pred_df_median=float(
            np.median(df_pred)
        ),
        pred_df_min=float(
            np.min(df_pred)
        ),
        pred_df_max=float(
            np.max(df_pred)
        ),
        sigma_floor=float(
            sigma_floor
        ),
        prev_raw_mean=float(
            np.mean(prev_raw)
        ),
        prev_raw_std=float(
            np.std(prev_raw)
        ),
        prev_norm_mean=float(
            np.mean(prev_norm)
        ),
        prev_norm_std=float(
            np.std(prev_norm)
        ),
    )

    log_info(
        "前几个交易日覆盖数量",
        head=daily_count.head(10).to_dict(),
    )

    return result


# ============================================================
# 本地 / 云端 Notebook 测试入口
# ============================================================

if __name__ == "__main__":
    from bigmodule import M

    # 请确认这是你云端真实的 5min 表名。
    # 如果不同，改这里，或通过环境变量 BAR5M_TABLE 覆盖。
    datasources = {
        "bar5m": os.environ.get(
            "BAR5M_TABLE",
            "bigalpha_2026_stock_bar5m",
        ),
    }

    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    log_info(
        "计算本地训练 DeepAR 5min 模型在云端数据上的 mu_over_sigma 分数",
        start=start_date,
        end=end_date,
        model_path=MODEL_PATH,
        datasources=datasources,
    )

    score_data = main(
        datasources=datasources,
        start_date=start_date,
        end_date=end_date,
    )

    print(score_data.head())
    print(score_data.tail())
    print(
        score_data.groupby("date")["instrument"]
        .nunique()
        .head(10)
    )

    log_info(
        "开始官方评估"
    )

    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        start_date=start_date,
        end_date=end_date,
        show=True,
    )
